# param-grad-access — worked example 1: manual SGD reading p.grad with None-guard

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `param-grad-access`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Every optimizer step reads the `.grad` tensor PyTorch populated on each parameter. The canonical loop iterates `model.parameters()`, skips any `p.grad is None` (a param that saw no forward pass), and applies `p.data -= lr * p.grad` in place so the update itself builds no autograd history.

## Worked solution

We implement `manual_sgd_step(params, lr)`. For each parameter we guard against `p.grad is None` with `continue`, because subtracting `None` from a tensor raises and an unused param simply has no gradient. Otherwise we update `p.data` in place, using `.data` to bypass autograd tracking. We also write `zero_grads` that sets each `p.grad = None`, the memory-cheap set-to-none strategy. We build two parameters, attach known gradients by hand (no backward needed), snapshot the originals, run the step, and print that each moved by exactly `-lr * grad`.

In [ ]:
import torch as t

t.manual_seed(0)

def manual_sgd_step(params, lr):
    for p in params:
        if p.grad is None:
            continue
        p.data -= lr * p.grad

def zero_grads(params):
    for p in params:
        p.grad = None

a = t.nn.Parameter(t.randn(3))
b = t.nn.Parameter(t.randn(3))
a.grad = t.ones(3)
b.grad = t.full((3,), 2.0)
a0 = a.data.clone()
manual_sgd_step([a, b], lr=0.1)
print('a delta:', (a.data - a0))  # -0.1 each